# Within-Cell-Type Theta Analysis

Compute within-cell-type vs marginal (across all cells) MoM theta for 3 datasets to empirically estimate the biological variance fraction (bio_frac).

Key question: is the default `dispersion_init_bio_frac=0.9` too low? With 100s of cell types, between-cell-type biological variance dominates and marginal excess overstates within-cell-type technical variance.


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from regularizedvi._dispersion_init import compute_dispersion_init

## Dataset paths and label keys

In [ ]:
# Dataset config: path, label_key, counts layer (None = X)
DATASETS = {
    "immune": {
        "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration_rna_v2/immune_adata_filtered.h5ad",
        "label_key": "harmonized_annotation",
        "layer": None,
    },
    "bone_marrow": {
        "path": "/nfs/team205/vk7/sanger_projects/cell2state/notebooks/bone_marrow_multimodal/results/bm_rna_filtered.h5ad",
        "label_key": "l2_cell_type",
        "layer": None,
    },
    "embryo": {
        "path": "/nfs/team205/vk7/sanger_projects/cell2state_embryo/results/embryo_rna_filtered.h5ad",
        "label_key": "cell_type_lvl5",
        "layer": None,
    },
}

# NOTE: update paths to match your environment if needed

## Compute per-cell-type theta for each dataset

In [ ]:
results = {}

for name, cfg in DATASETS.items():
    print(f"\n=== {name} ===")
    try:
        adata = sc.read_h5ad(cfg["path"])
    except FileNotFoundError as e:
        print(f"  SKIP: {e}")
        continue
    print(f"  Loaded: {adata.n_obs} cells x {adata.n_vars} genes")
    print(f"  Label key: {cfg['label_key']}")
    n_labels = adata.obs[cfg["label_key"]].nunique()
    print(f"  Unique labels: {n_labels}")

    log_theta, diag = compute_dispersion_init(
        adata,
        layer=cfg["layer"],
        biological_variance_fraction=0.9,
        label_key=cfg["label_key"],
        min_cells_per_group=50,
        verbose=True,
    )
    results[name] = {"log_theta": log_theta, "diag": diag}

## Compute empirical bio_frac per gene

For each gene g:
- `excess_marginal_g = var_g_all - mean_g_all` (from Poisson-subtracted marginal)
- `excess_within_g = mean_k(var_gk - mean_gk)` (weighted mean across cell types, weighted by mean_gk > 0)
- `bio_frac_g = 1 - excess_within_g / excess_marginal_g` — fraction of excess variance that is between-cell-type

In [ ]:
def compute_empirical_bio_frac(diag, eps=1e-6):
    """Compute per-gene empirical bio_frac from per-group stats."""
    mean_g = diag["mean_g"]
    var_g = diag["var_g"]
    excess_marginal = np.maximum(var_g - mean_g, eps)

    if "mean_g_per_group" not in diag:
        return None

    groups = list(diag["mean_g_per_group"].keys())
    # Stack per-group means and vars: (n_groups, n_genes)
    mean_gk = np.stack([diag["mean_g_per_group"][g] for g in groups], axis=0).astype(np.float64)
    var_gk = np.stack([diag["var_g_per_group"][g] for g in groups], axis=0).astype(np.float64)
    excess_gk = np.maximum(var_gk - mean_gk, eps)

    # Weighted mean across cell types, weighted by mean_gk > 0 AND group size
    group_sizes = np.array([diag["group_sizes"][g] for g in groups], dtype=np.float64)
    weights = (mean_gk > 0).astype(np.float64) * group_sizes[:, None]
    weight_sum = weights.sum(axis=0) + eps
    excess_within = (weights * excess_gk).sum(axis=0) / weight_sum

    bio_frac = 1.0 - excess_within / excess_marginal
    # For genes where within-type excess already exceeds marginal (noise), clamp to [0, 1]
    bio_frac = np.clip(bio_frac, 0.0, 1.0)

    # Within-cell-type theta (weighted mean of theta across groups, where defined)
    theta_gk = np.where(excess_gk > 0, (mean_gk**2) / excess_gk, np.nan)
    theta_within = np.nanmean(theta_gk, axis=0)

    return {
        "mean_g": mean_g,
        "excess_marginal": excess_marginal,
        "excess_within": excess_within,
        "bio_frac_g": bio_frac,
        "theta_within_g": theta_within,
        "theta_marginal_g": (mean_g**2) / excess_marginal,
    }


empirical = {}
for name in results:
    emp = compute_empirical_bio_frac(results[name]["diag"])
    if emp is None:
        continue
    empirical[name] = emp
    print(f"\n=== {name} ===")
    print(f"  bio_frac_g: median={np.nanmedian(emp['bio_frac_g']):.4f}, mean={np.nanmean(emp['bio_frac_g']):.4f}")
    print(
        f"  theta_marginal median={np.nanmedian(emp['theta_marginal_g']):.3f}, theta_within median={np.nanmedian(emp['theta_within_g']):.3f}"
    )
    ratio = emp["theta_within_g"] / (emp["theta_marginal_g"] + 1e-8)
    print(f"  theta_within / theta_marginal: median={np.nanmedian(ratio):.3f}, mean={np.nanmean(ratio):.3f}")

## Diagnostic plots

In [ ]:
n_datasets = len(empirical)
if n_datasets == 0:
    print("No datasets loaded — cannot plot")
else:
    fig, axes = plt.subplots(n_datasets, 4, figsize=(18, 4 * n_datasets), squeeze=False)
    for i, (name, emp) in enumerate(empirical.items()):
        # Col 0: hist of bio_frac_g
        ax = axes[i, 0]
        ax.hist(emp["bio_frac_g"], bins=50, color="steelblue", edgecolor="black")
        ax.axvline(0.9, color="red", linestyle="--", label="default=0.9")
        ax.axvline(0.99, color="orange", linestyle="--", label="alt=0.99")
        ax.axvline(
            np.nanmedian(emp["bio_frac_g"]),
            color="green",
            linestyle="-",
            label=f"median={np.nanmedian(emp['bio_frac_g']):.3f}",
        )
        ax.set_xlabel("empirical bio_frac_g")
        ax.set_ylabel("gene count")
        ax.set_title(f"{name}: bio_frac distribution")
        ax.legend(fontsize=8)

        # Col 1: hist2d theta_marginal vs theta_within
        ax = axes[i, 1]
        valid = (
            np.isfinite(emp["theta_marginal_g"])
            & np.isfinite(emp["theta_within_g"])
            & (emp["theta_marginal_g"] > 0)
            & (emp["theta_within_g"] > 0)
        )
        if valid.sum() > 10:
            ax.hist2d(
                np.log10(emp["theta_marginal_g"][valid]),
                np.log10(emp["theta_within_g"][valid]),
                bins=50,
                cmap="viridis",
            )
            lim = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
            ax.plot(lim, lim, "r--", lw=1, label="y=x")
        ax.set_xlabel("log10 theta_marginal")
        ax.set_ylabel("log10 theta_within")
        ax.set_title(f"{name}: marginal vs within-type theta")
        ax.legend(fontsize=8)

        # Col 2: hist2d gene_mean vs bio_frac_g
        ax = axes[i, 2]
        valid = np.isfinite(emp["bio_frac_g"]) & (emp["mean_g"] > 0)
        if valid.sum() > 10:
            ax.hist2d(
                np.log10(emp["mean_g"][valid] + 1e-6),
                emp["bio_frac_g"][valid],
                bins=50,
                cmap="viridis",
            )
        ax.set_xlabel("log10 gene_mean")
        ax.set_ylabel("bio_frac_g")
        ax.set_title(f"{name}: bio_frac vs expression")

        # Col 3: per-cell-type theta boxplot
        ax = axes[i, 3]
        diag = results[name]["diag"]
        if "theta_per_group" in diag:
            groups = sorted(diag["theta_per_group"].keys(), key=lambda g: -diag["group_sizes"][g])[:20]
            data = [np.log10(diag["theta_per_group"][g][diag["theta_per_group"][g] > 0]) for g in groups]
            ax.boxplot(data, showfliers=False)
            ax.set_xticklabels([f"{g[:8]}\n(n={diag['group_sizes'][g]})" for g in groups], rotation=90, fontsize=6)
            ax.set_ylabel("log10 theta_within")
            ax.set_title(f"{name}: theta by cell type (top 20)")

    plt.tight_layout()
    plt.show()

## Summary: empirical bio_frac per dataset

Use the median `bio_frac_g` as the recommended `dispersion_init_bio_frac` for each dataset.

In [ ]:
summary = pd.DataFrame(
    {
        name: {
            "n_cells": results[name]["diag"]["mean_g"].shape[0] if name in results else np.nan,
            "n_genes": len(emp["mean_g"]),
            "n_cell_types": len(results[name]["diag"].get("group_sizes", {})),
            "median_bio_frac": np.nanmedian(emp["bio_frac_g"]),
            "mean_bio_frac": np.nanmean(emp["bio_frac_g"]),
            "median_theta_marginal": np.nanmedian(emp["theta_marginal_g"]),
            "median_theta_within": np.nanmedian(emp["theta_within_g"]),
            "theta_ratio": np.nanmedian(emp["theta_within_g"] / (emp["theta_marginal_g"] + 1e-8)),
        }
        for name, emp in empirical.items()
    }
).T
print(summary)